In [2]:
import numpy as np
from common.util import *


In [3]:
x1 = np.random.rand(1, 3, 7, 7)
col1 = im2col(x1, 5, 5, stride=1, pad=0)
print(col1.shape)

x2 = np.random.rand(10, 3, 7, 7)
col2 = im2col(x2, 5, 5, stride=1, pad=0)
print(col2.shape)

(9, 75)
(90, 75)


In [4]:
# 利用im2col来实现卷积层
class Convolution:
    def __init__(self, W, b, stride=1, pad=0):
        self.W = W
        self.b = b
        self.stride = stride
        self.pad = pad

        # 保存前向计算的中间结果
        self.x = None
        self.col = None
        self.col_W = None

        # 保存参数的梯度
        self.dW = None
        self.db = None

    def forward(self, x):
        FN, C, FH, FW = self.W.shape
        N, C, H, W = x.shape
        out_h = int(1 + (H + 2 * self.pad - FH) / self.stride)
        out_w = int(1 + (W + 2 * self.pad - FW) / self.stride)

        col = im2col(x, FH, FW, self.stride, self.pad)
        # 比如， (10 , 3 , 5 , 5) 形状的数组的元素个数共有 750 个，指定 reshape(10,-1) 后，就 会转换成 (10 , 75) 形状的数组
        col_W = self.W.reshape(FN, -1).T
        out = np.dot(col, col_W) + self.b

        # transpose 通过指定索引 （编号） ，更改轴的顺序
        out = out.reshape(N, out_h, out_w, -1).transpose(0, 3, 1, 2)

        self.x = x
        self.col = col
        self.col_W = col_W

        return out

    def backward(self, dout):
        FN, C, FH, FW = self.W.shape

        # 把输出梯度恢复成矩阵乘法结果的形状
        dout = dout.transpose(0, 2, 3, 1).reshape(-1, FN)

        # 偏置梯度：把所有样本、所有空间位置的贡献相加
        self.db = np.sum(dout, axis=0)
        dcol_W = np.dot(self.col.T, dout)

        # 恢复成卷积核原来的形状
        self.dW = dcol_W.T.reshape(FN, C, FH, FW)

        # 展开后的输入梯度
        dcol = np.dot(dout, self.col_W.T)

        # 把各窗口的梯度累加回原输入位置
        dx = col2im(dcol, self.x.shape, FH, FW, self.stride, self.pad)

        return dx


In [5]:
# 池化层的实现
class Pooling:
    def __init__(self, pool_h, pool_w, stride=1, pad=0):
        self.pool_h = pool_h
        self.pool_w = pool_w
        self.stride = stride
        self.pad = pad

        self.x = None
        self.arg_max = None

    def forward(self, x):
        N, C, H, W = x.shape
        out_h = int(1 + (H - self.pool_h) / self.stride)
        out_w = int(1 + (W - self.pool_w) / self.stride)

        # 展开
        col = im2col(x, self.pool_h, self.pool_w, self.stride, self.pad)
        col = col.reshape(-1, self.pool_h * self.pool_w)

        # 记录每个窗口最大值所在的位置
        self.arg_max = np.argmax(col, axis=1)
        self.x = x

        # 最大值
        out = np.max(col, axis=1)
        # 转换
        out = out.reshape(N, out_h, out_w, C).transpose(0, 3, 1, 2)

        return out

    def backward(self, dout):
        # 调整顺序，与前向展开窗口的排列对应
        dout = dout.transpose(0, 2, 3, 1)

        pool_size = self.pool_h * self.pool_w

        # 每行对应一个池化窗口，先把梯度全部设为零
        dmax = np.zeros((dout.size, pool_size))

        # 只在最大值的位置填入该窗口收到的梯度
        dmax[np.arange(self.arg_max.size), self.arg_max] = dout.flatten()

        # 合并同一空间位置上的各通道窗口
        dcol = dmax.reshape(
            dout.shape[0] * dout.shape[1] * dout.shape[2], -1
        )

        # 将窗口梯度累加回原输入位置
        dx = col2im(
            dcol, self.x.shape,
            self.pool_h, self.pool_w,
            self.stride, self.pad
        )

        return dx

In [6]:
from collections import OrderedDict

from common.layers import Affine, Relu, SoftmaxWithLoss


class SimpleConvNet:
    def __init__(
        self,
        input_dim=(1, 28, 28),
        conv_param=None,
        hidden_size=100,
        output_size=10,
        weight_init_std=0.01,
    ):
        if conv_param is None:
            conv_param = {"filter_num": 30, "filter_size": 5, "pad": 0, "stride": 1}
        filter_num = conv_param["filter_num"]
        filter_size = conv_param["filter_size"]
        filter_pad = conv_param["pad"]
        filter_stride = conv_param["stride"]
        input_size = input_dim[1]
        conv_output_size = (
            input_size - filter_size + 2 * filter_pad
        ) / filter_stride + 1
        pool_output_size = int(
            filter_num * (conv_output_size / 2) * (conv_output_size / 2)
        )

        self.params = {}
        self.params["W1"] = weight_init_std * np.random.randn(
            filter_num, input_dim[0], filter_size, filter_size
        )
        self.params["b1"] = np.zeros(filter_num)
        self.params["W2"] = weight_init_std * np.random.randn(
            pool_output_size, hidden_size
        )
        self.params["b2"] = np.zeros(hidden_size)
        self.params["W3"] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params["b3"] = np.zeros(output_size)

        self.layers = OrderedDict()
        self.layers["Conv1"] = Convolution(
            self.params["W1"],
            self.params["b1"],
            conv_param["stride"],
            conv_param["pad"],
        )
        self.layers["Relu1"] = Relu()
        self.layers["Pool1"] = Pooling(pool_h=2, pool_w=2, stride=2)
        self.layers["Affine1"] = Affine(self.params["W2"], self.params["b2"])
        self.layers["Relu2"] = Relu()
        self.layers["Affine2"] = Affine(self.params["W3"], self.params["b3"])
        self.last_layer = SoftmaxWithLoss()

    def predict(self, x):
        for layer in self.layers.values():
            x = layer.forward(x)
        return x

    def loss(self, x, t):
        y = self.predict(x)
        return self.last_layer.forward(y, t)

    def gradient(self, x, t):
        # forward
        self.loss(x, t)

        # backward
        dout = 1
        dout = self.last_layer.backward(dout)

        layers = list(self.layers.values())
        layers.reverse()
        for layer in layers:
            dout = layer.backward(dout)

        # 设定
        grads = {}
        grads["W1"] = self.layers["Conv1"].dW
        grads["b1"] = self.layers["Conv1"].db
        grads["W2"] = self.layers["Affine1"].dW
        grads["b2"] = self.layers["Affine1"].db
        grads["W3"] = self.layers["Affine2"].dW
        grads["b3"] = self.layers["Affine2"].db

        return grads

In [8]:
def accuracy(network, x, t, batch_size=100):
    correct = 0

    for start in range(0, x.shape[0], batch_size):
        x_batch = x[start:start + batch_size]
        t_batch = t[start:start + batch_size]

        scores = network.predict(x_batch)
        predicted = np.argmax(scores, axis=1)
        actual = np.argmax(t_batch, axis=1)

        correct += np.sum(predicted == actual)

    return correct / x.shape[0]

In [ ]:
from common.mnist import load_mnist

(x_train, t_train), (x_test, t_test) = load_mnist(
    normalize=True,
    flatten=False,
    one_hot_label=True,
)

# x_train, t_train = x_train[:1000], t_train[:1000]
# x_test, t_test = x_test[:1000], t_test[:1000]

# 重新初始化，开始本次训练
network = SimpleConvNet()

epochs = 20
batch_size = 100
learning_rate = 0.1

train_loss_list = []
train_acc_list = []
test_acc_list = []

train_size = x_train.shape[0]

for epoch in range(epochs):
    indices = np.random.permutation(train_size)

    for start in range(0, train_size, batch_size):
        batch_mask = indices[start:start + batch_size]
        x_batch = x_train[batch_mask]
        t_batch = t_train[batch_mask]

        # 1. 前向与反向传播，得到参数梯度
        grads = network.gradient(x_batch, t_batch)

        # 2. SGD 更新参数
        for key in network.params:
            network.params[key] -= learning_rate * grads[key]

        # 3. 记录更新后的当前 batch 损失
        loss = network.loss(x_batch, t_batch)
        train_loss_list.append(loss)

    # 每完成一轮，再评价训练集与测试集
    train_acc = accuracy(network, x_train, t_train)
    test_acc = accuracy(network, x_test, t_test)

    train_acc_list.append(train_acc)
    test_acc_list.append(test_acc)

    print(
        f"epoch {epoch + 1}/{epochs} | "
        f"batch loss={loss:.4f} | "
        f"train acc={train_acc:.4f} | "
        f"test acc={test_acc:.4f}"
    )

epoch 1/50 | batch loss=0.2299 | train acc=0.9177 | test acc=0.9173
epoch 2/50 | batch loss=0.1055 | train acc=0.9596 | test acc=0.9578
epoch 3/50 | batch loss=0.0429 | train acc=0.9639 | test acc=0.9608
epoch 4/50 | batch loss=0.0868 | train acc=0.9785 | test acc=0.9750
epoch 5/50 | batch loss=0.0340 | train acc=0.9824 | test acc=0.9794
epoch 6/50 | batch loss=0.0097 | train acc=0.9855 | test acc=0.9815
epoch 7/50 | batch loss=0.0092 | train acc=0.9875 | test acc=0.9830
epoch 8/50 | batch loss=0.0327 | train acc=0.9903 | test acc=0.9847
epoch 9/50 | batch loss=0.0077 | train acc=0.9919 | test acc=0.9857
epoch 10/50 | batch loss=0.0078 | train acc=0.9926 | test acc=0.9857
epoch 11/50 | batch loss=0.0071 | train acc=0.9946 | test acc=0.9859
epoch 12/50 | batch loss=0.0031 | train acc=0.9934 | test acc=0.9852
epoch 13/50 | batch loss=0.0033 | train acc=0.9938 | test acc=0.9857
epoch 14/50 | batch loss=0.0058 | train acc=0.9950 | test acc=0.9855
epoch 15/50 | batch loss=0.0058 | train acc

KeyboardInterrupt: 